In [ ]:
!apt-get install openjdk-17-jre-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
!wget -q https://neo4j.com/artifact.php?name=neo4j-community-5.21.0-unix.tar.gz -O neo4j.tar.gz
!tar -xf neo4j.tar.gz  # or --strip-components=1
!mv neo4j-community-5.21.0 nj
!wget -q https://github.com/neo4j/apoc/releases/download/5.21.0/apoc-5.21.0-core.jar -O nj/plugins/apoc-5.21.0-core.jar
!echo "dbms.security.procedures.unrestricted=apoc.*" >> nj/conf/neo4j.conf
!echo "dbms.security.procedures.allowlist=apoc.*" >> nj/conf/neo4j.conf
!sed -i '/#dbms.security.auth_enabled/s/^#//g' nj/conf/neo4j.conf
!nj/bin/neo4j restart

Neo4j is not running.
Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:3526). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
%%capture
!pip install langchain-openai langchain-experimental -q
!pip install llama-index -q
!pip install llama-index-llms-azure-openai -q
!pip install llama-index-embeddings-azure-openai -q
!pip install llama-index-vector-stores-neo4jvector -q
!pip install llama-index-graph-stores-neo4j -q

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from google.colab import userdata
from llama_index.core import Settings

llm = AzureOpenAI(
    engine="gpt-4o",
    deployment_name=userdata.get("AZURE_MODEL_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
    temperature=0.0,
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    deployment_name=userdata.get("AZURE_EMBEDDING_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
)
llm.model = userdata.get("AZURE_MODEL_NAME")
embed_model.model_name = userdata.get("AZURE_EMBEDDING_NAME")
Settings.llm = llm
Settings.embed_model = embed_model

check: https://docs.llamaindex.ai/en/stable/examples/structured_outputs/structured_outputs/

In [ ]:
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from langchain_core.prompts import ChatPromptTemplate
from typing import Optional, List, Tuple
from pydantic import BaseModel, Field
from google.colab import userdata
import json
import requests

azure_llm = AzureChatOpenAI(
    model=userdata.get('AZURE_MODEL_NAME'),
    deployment_name=userdata.get('AZURE_MODEL_NAME'),
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=userdata.get('AZURE_API_KEY'),
    azure_endpoint=userdata.get('AZURE_BASE_URL'),
    api_version=userdata.get('AZURE_API_VERSION'),
)

https://www.cs.cmu.edu/~spok/grimmtmp/

In [ ]:
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore

pg_store = Neo4jPropertyGraphStore(
    username="neo4j",
    password="",
    url="bolt://localhost:7687",
)
pg_store.structured_query("""
MATCH (m)
DETACH DELETE m
""")

[]

In [ ]:
class GraphClasses(BaseModel):
    topics: List[str] = Field(description="A list genre of the text")
    # entities_classes: List[str] = Field(description="A list of all relevant entity classes in upper case")
    # relations_classes: List[str] = Field(description="A list of all relevant relation classes among the list of entities classes in upper case")
    relation_tuple_list: List[List[str]] = Field(description="""
    A list of tuples with contains: the entitySource, the entitySource name, the relation, the entityTarget, the entityTarget name""")
    insight_questions: List[str] = Field(description="A list of interesting questions that could has extensive response depending on the genres",
                                         min_length=10, max_length=20)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are a helpful assistant that extract a entities list depending the text topics
            Constraints:
            - The entities must be in upper case.
            - Ensure the tuples containt existing entities mentioned
            """,
        ),
        ("human", "<text>{text}</text>"),
    ]
)

chain = prompt | azure_llm.with_structured_output(GraphClasses)

In [ ]:
def fetch_text_from_url(url):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.text
        else:
            return f"Failed to retrieve the text. Status code: {response.status_code}"
    except Exception as e:
        return f"An error occurred: {e}"

url = "https://www.cs.cmu.edu/~spok/grimmtmp/027.txt"
text_content = fetch_text_from_url(url)

In [ ]:
graph_classes = chain.invoke({ "text": text_content })

In [ ]:
graph_classes

GraphClasses(topics=['FAIRY TALE', 'ANIMALS', 'MAGIC', 'FAMILY'], relation_tuple_list=[['TAILOR', 'TAILOR', 'HAS_SONS', 'SONS', 'SONS'], ['TAILOR', 'TAILOR', 'HAS_ANIMAL', 'GOAT', 'GOAT'], ['SONS', 'SONS', 'TAKE_CARE_OF', 'GOAT', 'GOAT'], ['GOAT', 'GOAT', 'PROVIDES', 'MILK', 'MILK'], ['SONS', 'SONS', 'TAKE_TURNS', 'PASTURE', 'PASTURE'], ['TAILOR', 'TAILOR', 'HAS_ITEM', 'YARD-MEASURE', 'YARD-MEASURE'], ['SONS', 'SONS', 'LEAVE_HOME', 'TAILOR', 'TAILOR'], ['SONS', 'SONS', 'LEARN_TRADE', 'JOINER', 'JOINER'], ['SONS', 'SONS', 'LEARN_TRADE', 'MILLER', 'MILLER'], ['SONS', 'SONS', 'LEARN_TRADE', 'TURNER', 'TURNER'], ['JOINER', 'JOINER', 'HAS_ITEM', 'MAGIC_TABLE', 'MAGIC_TABLE'], ['MILLER', 'MILLER', 'HAS_ITEM', 'GOLD-ASS', 'GOLD-ASS'], ['TURNER', 'TURNER', 'HAS_ITEM', 'CUDGEL_IN_SACK', 'CUDGEL_IN_SACK'], ['INNKEEPER', 'INNKEEPER', 'STEALS', 'MAGIC_TABLE', 'MAGIC_TABLE'], ['INNKEEPER', 'INNKEEPER', 'STEALS', 'GOLD-ASS', 'GOLD-ASS'], ['TURNER', 'TURNER', 'RETRIEVES', 'MAGIC_TABLE', 'MAGIC_TABLE'

In [ ]:
# class Neo4jIngester(BaseModel):
#     python_script: str = Field(description="A python script to ingest data to neo4j")

# prompt = ChatPromptTemplate.from_messages(
#     [
#         (
#             "system",
#             """You are a helpful assistant that generate a python script to ingest data to neo4j,
#             Constrains:
#             - depending of user graph candidates: relation_tuple_list.
#             - Assume the pg_store exist, thus do not add `pg_store = ... `
#             - Assume the chunk_text exist, thus do not add `chunk_text = ... `, do not declare chunk_text variable.
#             - Put the schema explicity.
#             - Use for instead of for inline [ for .. in ..].
#             - Use the following documentation:
# <python_documentaion>
# from llama_index.core.graph_stores.types import EntityNode, ChunkNode, Relation
# from llama_index.core.schema import TextNode

# # Create a two entity nodes
# entity1 = EntityNode(label="PERSON", name="Logan", properties={{"age": 28}})
# entity2 = EntityNode(label="ORGANIZATION", name="LlamaIndex")

# # Create a relation
# relation = Relation(
#     label="WORKS_FOR",
#     source_id=entity1.id,
#     target_id=entity2.id,
#     properties={{"since": 2023}},
# )

# pg_store.upsert_nodes([entity1, entity2])
# pg_store.upsert_relations([relation])

# source_node = TextNode(text=chunk_text)
# relations = [
#     Relation(
#         label="MENTIONS",
#         target_id=entity1.id,
#         source_id=source_node.node_id,
#     ),
#     Relation(
#         label="MENTIONS",
#         target_id=entity2.id,
#         source_id=source_node.node_id,
#     ),
# ]

# pg_store.upsert_llama_nodes([source_node])
# pg_store.upsert_relations(relations)
# </python_documentaion>

#             """,
#         ),
#         ("human", "Graph Classes:{relation_tuple_list}"),
#     ]
# )

# chain = prompt | azure_llm.with_structured_output(Neo4jIngester)

In [ ]:
# from llama_index.core.graph_stores.types import EntityNode, ChunkNode, Relation
# from llama_index.core.schema import TextNode
# neo4jIngester = chain.invoke({ "relation_tuple_list": str(graph_classes.relation_tuple_list) })

In [ ]:
# print(neo4jIngester.python_script)

In [ ]:
# from langchain_experimental.tools.python.tool import PythonAstREPLTool
# python_repl = PythonAstREPLTool(locals={"pg_store":  pg_store, "chunk_text":text_content, "relation_tuple_list":graph_classes.relation_tuple_list})
# python_repl.run(neo4jIngester.python_script)

In [ ]:
from llama_index.core.graph_stores.types import EntityNode, Relation
from llama_index.core.schema import TextNode

def process_graph_data(text_content, pg_store, graph_classes):
    # Create entity nodes
    entities = {}
    for relation_tuple in graph_classes.relation_tuple_list:
        source_label, source_name, relation_label, target_label, target_name = relation_tuple
        if source_name not in entities:
            entities[source_name] = EntityNode(label=source_label, name=source_name)
        if target_name not in entities:
            entities[target_name] = EntityNode(label=target_label, name=target_name)

    # Upsert entity nodes
    pg_store.upsert_nodes(list(entities.values()))

    # Create and upsert relations
    relations = []
    for relation_tuple in graph_classes.relation_tuple_list:
        source_name, _, relation_label, target_name, _ = relation_tuple
        relation = Relation(
            label=relation_label,
            source_id=entities[source_name].id,
            target_id=entities[target_name].id,
        )
        relations.append(relation)

    pg_store.upsert_relations(relations)

    # Create and upsert source text node
    source_node = TextNode(text=text_content)
    pg_store.upsert_llama_nodes([source_node])

    # Create and upsert mention relations
    mention_relations = []
    for entity in entities.values():
        mention_relation = Relation(
            label="MENTIONS",
            target_id=entity.id,
            source_id=source_node.node_id,
        )
        mention_relation.id
        # print(mention_relation.json())
        mention_relations.append(mention_relation)

    pg_store.upsert_relations(mention_relations)

process_graph_data(text_content, pg_store, graph_classes)

In [ ]:
query = "match (n:`__Node__`) return DISTINCT labels(n)"
result = pg_store.structured_query(query)
result

[{'labels(n)': ['__Node__', '__Entity__', 'TAILOR']},
 {'labels(n)': ['__Node__', '__Entity__', 'SONS']},
 {'labels(n)': ['__Node__', '__Entity__', 'GOAT']},
 {'labels(n)': ['__Node__', '__Entity__', 'MILK']},
 {'labels(n)': ['__Node__', '__Entity__', 'PASTURE']},
 {'labels(n)': ['__Node__', '__Entity__', 'YARD-MEASURE']},
 {'labels(n)': ['__Node__', '__Entity__', 'JOINER']},
 {'labels(n)': ['__Node__', '__Entity__', 'MILLER']},
 {'labels(n)': ['__Node__', '__Entity__', 'TURNER']},
 {'labels(n)': ['__Node__', '__Entity__', 'MAGIC_TABLE']},
 {'labels(n)': ['__Node__', '__Entity__', 'FOOD']},
 {'labels(n)': ['__Node__', '__Entity__', 'GOLD-ASS']},
 {'labels(n)': ['__Node__', '__Entity__', 'GOLD']},
 {'labels(n)': ['__Node__', '__Entity__', 'CUDGEL_IN_SACK']},
 {'labels(n)': ['__Node__', '__Entity__', 'PROTECTION']},
 {'labels(n)': ['__Node__', '__Entity__', "FOX'S_HOLE"]},
 {'labels(n)': ['__Node__', '__Entity__', 'FOX']},
 {'labels(n)': ['__Node__', '__Entity__', 'BEAR']},
 {'labels(n)'

In [ ]:
# query = "match (n:Chunk) return n"
# result = pg_store.structured_query(query)
# result

In [ ]:
pg_store.structured_query('''
MATCH (n)
WITH count(n) AS node_count
MATCH ()-[r]->()
RETURN node_count, count(r) AS relation_count
''')

[{'node_count': 20, 'relation_count': 42}]

In [ ]:
# print(chain.invoke({ "graph_classes": str(graph_classes) }).python_script)

In [ ]:
from tqdm.auto import tqdm
import time
i = 0
for node in tqdm(pg_store.structured_query("""
MATCH (node)
WHERE not node:Chunk and node.embedding IS NULL
WITH node, apoc.map.removeKeys(properties(node), ['triplet_source_id', 'id', 'embedding']) AS filtered_properties
RETURN elementId(node), labels(node)[-1] AS node_label, filtered_properties
""")[:]):
    (id, node_label, filtered_properties) = list(node.values())
    emd = embed_model.get_query_embedding(f"{node_label}:{filtered_properties}")
    pg_store.structured_query(f"""
    MATCH (c)
    WHERE elementId(c) = '{id}'
    SET c.embedding = {emd}
    """)
    i += 1
    if i%100 == 0:
        !nj/bin/neo4j restart > /dev/null 2>&1
        pg_store = Neo4jPropertyGraphStore( username="neo4j", password="", url="bolt://localhost:7687")

  0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
from llama_index.core.data_structs import Node
from llama_index.core.schema import NodeWithScore
from llama_index.core import get_response_synthesizer
from llama_index.core.response_synthesizers import ResponseMode
from llama_index.core import PropertyGraphIndex

response_synthesizer = get_response_synthesizer(llm,response_mode=ResponseMode.TREE_SUMMARIZE)
index = PropertyGraphIndex.from_existing(
    property_graph_store=pg_store,
    llm=llm,
    embed_model=embed_model,
)

In [ ]:
for i,q in enumerate(graph_classes.insight_questions,start=1):
    response = response_synthesizer.synthesize(q, nodes=index.as_query_engine(llm, similarity_top_k=50,path_depth=1).retrieve(q))
    print(f"Question({i}):",q)
    print("Answer:",response.response,"\n")

Question(1): WHAT IS THE SIGNIFICANCE OF THE GOAT IN THE STORY?
Answer: The goat in the story is significant because it provides milk, which is a valuable resource. Additionally, the goat's actions, such as hiding in the fox's hole and being driven out by the tailor, contribute to the unfolding events and interactions among the characters. 

Question(2): HOW DO THE SONS' TRADES CONTRIBUTE TO THE PLOT?
Answer: The sons' trades contribute to the plot by providing them with unique items that play significant roles. The joiner has a magic table that provides food, the miller has a gold-ass that provides gold, and the turner has a cudgel in a sack that provides protection. These items likely aid the sons in their adventures and challenges. 

Question(3): WHAT ROLE DOES THE MAGIC TABLE PLAY IN THE STORY?
Answer: The magic table provides food in the story. 

Question(4): HOW DOES THE GOLD-ASS AFFECT THE MILLER'S LIFE?
Answer: The Gold-Ass provides gold, which would likely have a significant p

In [ ]:
r= index.as_query_engine(llm, similarity_top_k=1,path_depth=1,include_text=False).retrieve("What happened to the GOAT at the end of the story?")
print(len(r))
r[0]

6


NodeWithScore(node=TextNode(id_='8dc180c2-c105-4719-a007-b51769776872', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='TAILOR -> HAS_ANIMAL -> GOAT', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.76578289270401)

In [ ]:
for node in r:
    print(node.text)

TAILOR -> HAS_ANIMAL -> GOAT
SONS -> TAKE_CARE_OF -> GOAT
GOAT -> PROVIDES -> MILK
TAILOR -> DRIVES_OUT -> GOAT
GOAT -> HIDES_IN -> FOX'S_HOLE
BEE -> STINGS -> GOAT


In [ ]:
chunks = set()

for node in r:
    start_node, relationship, end_node = node.text.split(' -> ')
    query = f"""MATCH (a:`{start_node}`)-[r:{relationship}]->(b:`{end_node}`)
MATCH (c:Chunk)-[m:MENTIONS]->(x)
WHERE x IN [a, b]
RETURN DISTINCT c.text as chunk"""
    result = pg_store.structured_query(query)
    for t in result:
        chunks.add(t['chunk'])

In [ ]:
len(list(chunks))

1

In [ ]:
for i,q in enumerate(graph_classes.insight_questions,start=1):
    print(f"Question({i}):",q)
    relations_result = index.as_query_engine(llm, similarity_top_k=1,path_depth=1,include_text=False).retrieve(q)
    chunks = set(node.text for node in relations_result)
    for node in relations_result:
        start_node, relationship, end_node = node.text.split(' -> ')
        query = f"""MATCH (a:`{start_node}`)-[r:{relationship}]->(b:`{end_node}`)
            MATCH (c:Chunk)-[m:MENTIONS]->(x) WHERE x IN [a, b] RETURN DISTINCT c.text as chunk"""
        for t in pg_store.structured_query(query):
            chunks.add(t['chunk'])
    answer = response_synthesizer.get_response(query_str=q, text_chunks=chunks)
    print("Answer:",answer,"\n")

Question(1): WHAT IS THE SIGNIFICANCE OF THE GOAT IN THE STORY?
Answer: The goat in the story serves as a catalyst for the events that unfold. Initially, the goat's false claims of not being fed properly lead the tailor to unjustly drive away his three sons. This sets off a chain of events where each son goes on to learn a trade and acquire magical items. Ultimately, the goat's deceit is revealed, and she is punished by the tailor. The goat's actions indirectly lead to the sons' adventures and eventual return with their magical gifts, which bring prosperity to the family. 

Question(2): HOW DO THE SONS' TRADES CONTRIBUTE TO THE PLOT?
Answer: The sons' trades play a crucial role in the plot by providing them with unique magical items that ultimately lead to the resolution of the story. The eldest son, who becomes a joiner, receives a table that can magically spread itself with food. The second son, who becomes a miller, is given a gold-spewing ass. The third son, who becomes a turner, i

In [ ]:
r = index.as_query_engine(llm, similarity_top_k=1,path_depth=1,include_text=True).query("What happened to the GOAT at the end of the story?")
r

Response(response='The goat was stung by a bee.', source_nodes=[NodeWithScore(node=TextNode(id_='6d595c2c-86b1-4bfc-941c-2feecb3d5f9b', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='TAILOR -> HAS_ANIMAL -> GOAT', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.76578289270401), NodeWithScore(node=TextNode(id_='4f42a7e0-2ab4-4527-951e-c60344a453a5', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='ELDEST SON -> TAKES_TO_PASTURE -> GOAT', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.76578289270401), NodeWithScore(node=TextNode(id_='c51c52f8-d299-4da7-a48f-5fc462884eda', embedding=None, metadata

In [ ]:
r = index.as_query_engine(llm, similarity_top_k=1,path_depth=1,include_text=False).query("What happened to the GOAT at the end of the story?")
r

Response(response="The GOAT hid in the FOX'S HOLE at the end of the story.", source_nodes=[NodeWithScore(node=TextNode(id_='87c8e768-6f73-4fcf-bec7-0276a21a0478', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text="GOAT -> HIDES_IN -> FOX'S HOLE", mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.76578289270401), NodeWithScore(node=TextNode(id_='7046f930-104e-46b0-b618-9f2d3aea37fc', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='BEE -> STINGS -> GOAT', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.76578289270401), NodeWithScore(node=TextNode(id_='cd5a8151-f131-403b-bc49-365aae5344ed', embedding=No

In [ ]:
pg_store.structured_query(f"""
MATCH (c:Chunk)-[r]->(m)
RETURN properties(r)
""")

[{'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}},
 {'properties(r)': {}}]

In [ ]:
q = "What happened to the GOAT at the end of the story?"

In [ ]:
print(dir(index.as_query_engine(llm, similarity_top_k=1,path_depth=1)))
index.as_retriever().retrieve(q)

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_apply_node_postprocessors', '_aquery', '_as_query_component', '_get_prompt_modules', '_get_prompts', '_node_postprocessors', '_query', '_response_synthesizer', '_retriever', '_update_prompts', '_validate_prompts', 'aquery', 'aretrieve', 'as_query_component', 'asynthesize', 'callback_manager', 'from_args', 'get_prompts', 'query', 'retrieve', 'retriever', 'synthesize', 'update_prompts', 'with_retriever']


[NodeWithScore(node=TextNode(id_='a69d0863-f353-4704-9052-726646ab6705', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text="GOAT -> HIDES_IN -> FOX'S HOLE", mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.76578289270401),
 NodeWithScore(node=TextNode(id_='24b4b576-9060-4586-ac46-940790caa45d', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, text='BEE -> STINGS -> GOAT', mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.76578289270401),
 NodeWithScore(node=TextNode(id_='92f84ad2-7266-42d0-95c4-2253d88e0cec', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relatio

In [ ]:
response_synthesizer.get_response(query_str=q,
                                #   nodes=index.as_query_engine(llm, similarity_top_k=1,path_depth=1).retrieve(q),
                                  text_chunks=chunks)

'At the end of the story, the goat, ashamed of her bald head, ran to a fox\'s hole and crept into it. When the fox returned and saw the goat\'s fiery eyes in the darkness, he was terrified and ran away. The fox then encountered a bear, who also became frightened upon seeing the goat\'s eyes and fled. Finally, a bee offered to help and flew into the fox\'s cave, stinging the goat on her shorn head. The goat sprang up, crying "meh, meh," and ran away into the world, never to be seen again.'

In [ ]:
type(chunks)

set